# 07 - `ml.trip_validity_final`: the final trip-validity table

The deliverable this whole model came from: one row per trip (all
940,988), with a single validity flag, sourced either from a human
label (the 500 collected via the labeling app) or from the notebook-06
final model - never both, never neither.

**Columns**: `trip_id`, `bus_id` (5-digit), `route_id` (already the
correct zero-padded internal format - see the note below), `trip_date`,
`trip_start_timestamp`/`trip_end_timestamp`, `gtfs_feed_version_date` +
`gtfs_shape_id_i`/`gtfs_shape_id_v` (a single join away from
`ml.trip_validity_route_shapes`/`ml.trip_validity_route_stops` for
route geometry and ordered stops - no need to go through
`ml.trip_validity_route_gtfs_match` first), `is_valid`, `label_source`
(`'human'` or `'model'`), and `model_name`/`model_confidence` (both
`NULL` for human-labeled rows).

**On `route_id` format**: checked directly - 935,004 of 940,988 trips
(99.36%) already have a clean 3-digit zero-padded code. The remaining
5,984 are genuinely 4-digit: `1074` and `1815` are real routes with
real GTFS matches (the agency itself uses 4 digits for them, confirmed
against `gtfs_route_short_name`); `1075` and `9999` have no GTFS match
in any feed at all (`9999` in particular looks like a sentinel/
placeholder in the source AFC data). `route_id` is carried over as-is -
forcing everything to exactly 3 digits would corrupt the two genuine
4-digit routes for no benefit.

**On the ~1.7% of trips with no GTFS match** (30 of 352 routes never
matched any feed, `trip_validity_route_gtfs_match.gtfs_feed_version_date
IS NULL`): included anyway, with `gtfs_feed_version_date`/
`gtfs_shape_id_i`/`gtfs_shape_id_v` left `NULL` - this is a table of
*trips*, not of trips-with-known-routes, so every trip gets a row
regardless of whether its route geometry happens to be resolvable.

**Model scoring**: every trip without a human label gets scored by the
notebook-06 final model (`ml/trip_validity_model/artifacts/
final_lightgbm_model.joblib` + its saved decision threshold), tagged
`model_name = 'trip_validity_lightgbm_v1'` - a stable version string,
not a file path, so it survives that artifact being retrained or moved
later without silently going stale.

In [1]:
import json
import os
import sys
from pathlib import Path

import joblib
import pandas as pd
import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))
sys.path.insert(0, str(_root / "ml/trip_validity_model/app"))

## Connect and import the app's own feature list and calibrator

In [3]:
import features
from calibration import PlattCalibrator

from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
print("connected")

connected


## Create the table

`is_valid`/`label_source`/`model_name`/`model_confidence` are
constrained together: a human row must have both model columns `NULL`;
a model row must have both `NOT NULL` - the schema itself enforces
"never both, never neither" rather than trusting every future INSERT
to get it right.

In [4]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_final CASCADE;

    CREATE TABLE ml.trip_validity_final (
        trip_id                 bigint NOT NULL
            REFERENCES ml.trip_validity_dataset (trip_id),
        bus_id                  text NOT NULL,
        route_id                text NOT NULL,
        trip_date                date NOT NULL,
        trip_start_timestamp    timestamptz NOT NULL,
        trip_end_timestamp      timestamptz NOT NULL,
        gtfs_feed_version_date  date,
        gtfs_shape_id_i         text,
        gtfs_shape_id_v         text,
        is_valid                boolean NOT NULL,
        label_source            text NOT NULL
            CHECK (label_source IN ('human', 'model')),
        model_name              text,
        model_confidence        double precision,
        PRIMARY KEY (trip_id),
        CHECK (
            (label_source = 'human' AND model_name IS NULL AND model_confidence IS NULL)
            OR
            (label_source = 'model' AND model_name IS NOT NULL
                AND model_confidence IS NOT NULL)
        )
    );
""")
conn.commit()
print("table created")

table created


## Insert the 500 human-labeled trips

Direct copy of the label - no model involved, so `model_name`/
`model_confidence` stay `NULL`.

In [5]:
conn.execute("""
    INSERT INTO ml.trip_validity_final (
        trip_id, bus_id, route_id, trip_date,
        trip_start_timestamp, trip_end_timestamp,
        gtfs_feed_version_date, gtfs_shape_id_i, gtfs_shape_id_v,
        is_valid, label_source, model_name, model_confidence
    )
    SELECT
        d.trip_id, d.bus_id, d.route_id, d.trip_date,
        d.trip_opening_timestamp, d.trip_closing_timestamp,
        d.gtfs_feed_version_date, d.gtfs_shape_id_i, d.gtfs_shape_id_v,
        l.label, 'human', NULL, NULL
    FROM ml.trip_validity_labels l
    JOIN ml.trip_validity_dataset d ON d.trip_id = l.trip_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_final;")
    print(f"{cur.fetchone()[0]} human-labeled rows inserted")

500 human-labeled rows inserted


## Score every remaining trip with the final model

Loads the notebook-06 final artifact (model, calibrator, selected
features) and its saved decision threshold, then predicts on every
trip *not* already in `ml.trip_validity_labels`.

In [6]:
MODEL_NAME = "trip_validity_lightgbm_v1"

ARTIFACT_DIR = _root / "ml/trip_validity_model/artifacts"
artifact = joblib.load(ARTIFACT_DIR / "final_lightgbm_model.joblib")
final_model = artifact["model"]
selected_features = artifact["selected_features"]
calibrator = PlattCalibrator.from_params(**artifact["calibrator_params"])

with (ARTIFACT_DIR / "final_model_summary.json").open() as f:
    model_summary = json.load(f)
DECISION_THRESHOLD = model_summary["decision_threshold"]

print(f"model: {MODEL_NAME}, {len(selected_features)} features")
print(f"decision threshold: {DECISION_THRESHOLD:.4f}")

model: trip_validity_lightgbm_v1, 80 features
decision threshold: 0.8454


### Fetch every trip without a human label

Pulls both the identifier/timestamp columns this table needs and every
feature the model was trained on, in one query.

In [7]:
feature_columns_sql = sql.SQL(", ").join(
    sql.SQL("d.{}").format(sql.Identifier(c)) for c in features.ALL_FEATURES
)

unlabeled_query = sql.SQL("""
    SELECT
        d.trip_id, d.bus_id, d.route_id, d.trip_date,
        d.trip_opening_timestamp, d.trip_closing_timestamp,
        d.gtfs_feed_version_date, d.gtfs_shape_id_i, d.gtfs_shape_id_v,
        {feature_columns}
    FROM ml.trip_validity_dataset d
    WHERE NOT EXISTS (
        SELECT 1 FROM ml.trip_validity_labels l WHERE l.trip_id = d.trip_id
    )
""").format(feature_columns=feature_columns_sql)

with conn.cursor() as cur:
    cur.execute(unlabeled_query)
    cols = [c.name for c in cur.description]
    unlabeled = pd.DataFrame(cur.fetchall(), columns=cols)

unlabeled = features.cast_feature_dtypes(unlabeled)
print(f"{len(unlabeled)} trips to score")

940488 trips to score


### Score, threshold, and bulk-insert

`model_confidence` is the calibrated P(valid) itself, not a
"confidence in whichever direction" transform - so a value near 0
means confidently invalid, near 1 confidently valid, near the
decision threshold means genuinely unsure. `is_valid` is just that
same value thresholded at `DECISION_THRESHOLD` (chosen in notebook 06
by maximizing F1 on the calibrated out-of-fold predictions, not a
default 0.5).

In [8]:
raw_proba = final_model.predict_proba(unlabeled[selected_features])[:, 1]
calibrated_proba = calibrator.predict(raw_proba)

model_rows = unlabeled[
    [
        "trip_id",
        "bus_id",
        "route_id",
        "trip_date",
        "trip_opening_timestamp",
        "trip_closing_timestamp",
        "gtfs_feed_version_date",
        "gtfs_shape_id_i",
        "gtfs_shape_id_v",
    ]
].copy()
model_rows["is_valid"] = [bool(v) for v in (calibrated_proba >= DECISION_THRESHOLD)]
model_rows["label_source"] = "model"
model_rows["model_name"] = MODEL_NAME
model_rows["model_confidence"] = [float(v) for v in calibrated_proba]
model_rows = model_rows.where(pd.notna(model_rows), None)

insert_columns = [
    "trip_id",
    "bus_id",
    "route_id",
    "trip_date",
    "trip_opening_timestamp",
    "trip_closing_timestamp",
    "gtfs_feed_version_date",
    "gtfs_shape_id_i",
    "gtfs_shape_id_v",
    "is_valid",
    "label_source",
    "model_name",
    "model_confidence",
]
table_columns = [
    "trip_id",
    "bus_id",
    "route_id",
    "trip_date",
    "trip_start_timestamp",
    "trip_end_timestamp",
    "gtfs_feed_version_date",
    "gtfs_shape_id_i",
    "gtfs_shape_id_v",
    "is_valid",
    "label_source",
    "model_name",
    "model_confidence",
]

copy_query = sql.SQL("COPY ml.trip_validity_final ({}) FROM STDIN").format(
    sql.SQL(", ").join(sql.Identifier(c) for c in table_columns)
)
with conn.cursor() as cur, cur.copy(copy_query) as copy:
    for row in model_rows[insert_columns].itertuples(index=False, name=None):
        copy.write_row(row)
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_final;")
    print(f"{cur.fetchone()[0]} total rows after model scoring")

940988 total rows after model scoring


## Indexes and comments

`(route_id, trip_date)` for resolving back through
`ml.trip_validity_route_gtfs_match`; `(gtfs_feed_version_date,
gtfs_shape_id_i/v)` for going straight to
`ml.trip_validity_route_shapes`/`ml.trip_validity_route_stops`;
`label_source` since "give me every model-scored trip" is an obvious
query.

In [9]:
conn.execute("""
    CREATE INDEX trip_validity_final_route_date_idx
        ON ml.trip_validity_final (route_id, trip_date);
    CREATE INDEX trip_validity_final_shape_i_idx
        ON ml.trip_validity_final (gtfs_feed_version_date, gtfs_shape_id_i);
    CREATE INDEX trip_validity_final_shape_v_idx
        ON ml.trip_validity_final (gtfs_feed_version_date, gtfs_shape_id_v);
    CREATE INDEX trip_validity_final_label_source_idx
        ON ml.trip_validity_final (label_source);
""")
conn.execute("ANALYZE ml.trip_validity_final;")
conn.commit()


def comment_on_column(cur: psycopg.Cursor, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one trip_validity_final column."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.trip_validity_final.{} IS {};").format(
            sql.Identifier(col), sql.Literal(text)
        )
    )


COLUMN_COMMENTS = {
    "trip_id": "ml.trip_validity_dataset.trip_id, as-is.",
    "bus_id": "ml.trip_validity_dataset.bus_id (5-digit AFC vehicle number), as-is.",
    "route_id": (
        "ml.trip_validity_dataset.route_id, as-is - already zero-padded "
        "(usually 3 digits; a small number of routes are genuinely "
        "4 digits, confirmed against gtfs_route_short_name, not a "
        "formatting artifact)."
    ),
    "trip_date": "ml.trip_validity_dataset.trip_date, as-is.",
    "trip_start_timestamp": "ml.trip_validity_dataset.trip_opening_timestamp, as-is.",
    "trip_end_timestamp": "ml.trip_validity_dataset.trip_closing_timestamp, as-is.",
    "gtfs_feed_version_date": (
        "ml.trip_validity_dataset.gtfs_feed_version_date, as-is. NULL if "
        "this trip's route never matched any GTFS feed."
    ),
    "gtfs_shape_id_i": (
        "ml.trip_validity_dataset.gtfs_shape_id_i, as-is - join to "
        "ml.trip_validity_route_shapes/route_stops on "
        "(gtfs_feed_version_date, gtfs_shape_id_i) for the I-direction "
        "route geometry/stops."
    ),
    "gtfs_shape_id_v": ("Same as gtfs_shape_id_i, for the V direction."),
    "is_valid": (
        "The final validity determination - a human label if one "
        "exists, otherwise the notebook-06 model's prediction "
        "thresholded at its own saved decision threshold."
    ),
    "label_source": (
        "'human' (500 rows, from the labeling app) or 'model' (everything else)."
    ),
    "model_name": (
        "Version string identifying which model scored this trip - "
        "NULL for human-labeled rows. See "
        "ml/trip_validity_model/notebooks/06_model_sweep.ipynb."
    ),
    "model_confidence": (
        "Calibrated P(valid) from that model - NOT a "
        "confidence-in-whichever-direction transform. Near 0 = "
        "confidently invalid, near 1 = confidently valid, near the "
        "decision threshold = genuinely unsure. NULL for human-labeled "
        "rows."
    ),
}
with conn.cursor() as cur:
    for col, text in COLUMN_COMMENTS.items():
        comment_on_column(cur, col, text)
    cur.execute(
        sql.SQL("COMMENT ON TABLE ml.trip_validity_final IS {};").format(
            sql.Literal(
                "The final Trip Validity deliverable: one row per trip, "
                "human-labeled where available, model-scored otherwise. "
                "See ml/trip_validity_model/notebooks/07_final_trip_table.ipynb."
            )
        )
    )
conn.commit()
print("indexes and comments applied")

indexes and comments applied


## Verification

In [10]:
EXPECTED_TOTAL = 940988
EXPECTED_HUMAN = 500

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            count(*) AS total,
            count(*) FILTER (WHERE label_source = 'human') AS human,
            count(*) FILTER (WHERE label_source = 'model') AS model,
            count(*) FILTER (WHERE is_valid) AS valid_count,
            count(*) FILTER (WHERE gtfs_feed_version_date IS NULL) AS no_feed_match,
            count(*) FILTER (
                WHERE label_source = 'model'
                  AND (model_name IS NULL OR model_confidence IS NULL)
            ) AS model_rows_missing_model_info,
            count(*) FILTER (
                WHERE label_source = 'human'
                  AND (model_name IS NOT NULL OR model_confidence IS NOT NULL)
            ) AS human_rows_with_model_info
        FROM ml.trip_validity_final;
    """)
    cols = [c.name for c in cur.description]
    summary = dict(zip(cols, cur.fetchone(), strict=True))
    print(summary)

    cur.execute("""
        SELECT count(*)
        FROM ml.trip_validity_final f
        JOIN ml.trip_validity_labels l ON l.trip_id = f.trip_id
        WHERE f.label_source = 'human' AND f.is_valid != l.label;
    """)
    mismatched_human_labels = cur.fetchone()[0]

    cur.execute("""
        SELECT count(DISTINCT route_id)
        FROM ml.trip_validity_final WHERE gtfs_feed_version_date IS NULL;
    """)
    orphan_route_count = cur.fetchone()[0]

if summary["total"] != EXPECTED_TOTAL:
    msg = f"expected {EXPECTED_TOTAL} total rows, got {summary['total']}"
    raise AssertionError(msg)
if summary["human"] != EXPECTED_HUMAN:
    msg = f"expected {EXPECTED_HUMAN} human-labeled rows, got {summary['human']}"
    raise AssertionError(msg)
if summary["model"] != EXPECTED_TOTAL - EXPECTED_HUMAN:
    msg = "human + model row counts don't add up to the total"
    raise AssertionError(msg)
if summary["model_rows_missing_model_info"] != 0:
    msg = "found model-sourced rows missing model_name/model_confidence"
    raise AssertionError(msg)
if summary["human_rows_with_model_info"] != 0:
    msg = "found human-sourced rows with non-NULL model_name/model_confidence"
    raise AssertionError(msg)
if mismatched_human_labels != 0:
    msg = f"{mismatched_human_labels} rows disagree with their own source label"
    raise AssertionError(msg)

print(
    f"OK: {summary['total']} total rows ({summary['human']} human, "
    f"{summary['model']} model), {summary['valid_count']} valid "
    f"({summary['valid_count'] / summary['total']:.1%}), "
    f"{summary['no_feed_match']} rows across {orphan_route_count} routes "
    "with no GTFS feed match"
)

{'total': 940988, 'human': 500, 'model': 940488, 'valid_count': 720080, 'no_feed_match': 6833, 'model_rows_missing_model_info': 0, 'human_rows_with_model_info': 0}
OK: 940988 total rows (500 human, 940488 model), 720080 valid (76.5%), 6833 rows across 30 routes with no GTFS feed match
